# 02 · Theory checks: EGS versus Shannon regularization
Numerical illustrations of SI Proposition 1 (bounded gradient), Theorem 1 (bounded curvature, conditioning
at Π = 0) and Theorem 2 (permutation invariance). The same statements are asserted in `tests/test_theory.py`.

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from vbgnn.regularizers import (egs_struct_loss, egs_struct_hessian, egs_hessian_bound,
                                egs_gradient_bound, shannon_struct_loss, structural_diversity)
rng = np.random.default_rng(0)
d, lam, beta = 5, 1.0, 1.0
A = rng.normal(size=(d, d)); Sigma = A @ A.T / d + 0.2 * np.eye(d)

# gradient of one coordinate as pi_ij -> 0, all other entries fixed
base = rng.uniform(0.2, 0.8, (d, d))
ps = np.logspace(-12, -0.31, 200)
g_egs, g_sh = [], []
for p in ps:
    P = base.copy(); P[0, 1] = p
    g_egs.append(abs(egs_struct_loss(P, Sigma, lam, beta, normalize=False)[1][0, 1]))
    g_sh.append(abs(shannon_struct_loss(P, lam, eps=1e-300)[1][0, 1]))
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.loglog(ps, g_sh, color="#eb6834", lw=2, label="Shannon: |∂H/∂π| = |ln((1-π)/π)|")
ax.loglog(ps, g_egs, color="#2a78d6", lw=2.4, label="EGS: |∂L_struct/∂π|")
ax.axhline(egs_gradient_bound(Sigma, lam, beta, d, normalize=False), color="#6b6a64", ls="--", lw=1,
           label="M_struct (Proposition 1)")
ax.set_xlabel("edge probability π_ij"); ax.set_ylabel("|gradient|"); ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
# Theorem 1(i): spectral norm of the Hessian never exceeds 12 λ β λmax(Σ) d²
norms = [np.linalg.norm(egs_struct_hessian(rng.uniform(size=(d, d)), Sigma, lam, beta, normalize=False), 2)
         for _ in range(500)]
print("max ||Hessian||_2 over 500 random Π: %.2f   bound: %.2f"
      % (max(norms), egs_hessian_bound(Sigma, lam, beta, d, normalize=False)))

# Theorem 1(ii): at Π = 0 the Hessian is 2λβ (I ⊗ Σ) and its condition number equals κ(Σ)
H0 = egs_struct_hessian(np.zeros((d, d)), Sigma, lam, beta, normalize=False)
print("κ(H(0)) = %.4f   κ(Σ) = %.4f" % (np.linalg.cond(H0), np.linalg.cond(Sigma)))

In [ ]:
# Theorem 2: invariance under node relabelling
P = rng.uniform(size=(d, d)); Pm = np.eye(d)[rng.permutation(d)]
print("D(Π) = %.6f,  D(PΠPᵀ) = %.6f" % (structural_diversity(P), structural_diversity(Pm @ P @ Pm.T)))
print("L(Π;Σ) = %.6f, L(PΠPᵀ; PΣPᵀ) = %.6f" % (egs_struct_loss(P, Sigma, lam, beta)[0],
                                                 egs_struct_loss(Pm @ P @ Pm.T, Pm @ Sigma @ Pm.T, lam, beta)[0]))